# THEMAP — OTDD deep dive

<a href="https://colab.research.google.com/github/HFooladi/THEMAP/blob/main/notebooks/colab/03_otdd_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[`01_quick_tour.ipynb`](01_quick_tour.ipynb) used ECFP fingerprints + Euclidean distance because both are fast and CPU-friendly. They are *not*, however, what THEMAP is built around. The headline distance method is **OTDD** (Optimal Transport Dataset Distance, Alvarez-Melis & Fusi 2020), which measures the Wasserstein distance between two datasets viewed as joint distributions over (feature, label) pairs. That makes the distance *label-aware* — datasets with similar molecules but flipped activities will look far apart, where Euclidean would see them as close.

This notebook answers three questions:

1. How do I actually compute OTDD between chemical datasets?
2. Why does OTDD need **continuous** molecular representations — and what happens when you feed it binary fingerprints?
3. How do OTDD, Euclidean, and Cosine compare on the same data?

> ⚠️ **GPU required.** OTDD is GPU-bound at any practical scale. In Colab: **Runtime → Change runtime type → T4 GPU** before running. Expect ~5–10 minutes end-to-end on a T4; the ChemBERTa featurizer downloads ~300 MB on first use.

## 1. Install

We install THEMAP with the `otdd` extra (which pulls in `pot`, `geomloss`, and `pykeops` on top of the base deps), `molfeat` (featurizers), `transformers` (for the ChemBERTa neural embedding — installing it directly lets pip pull a current `tokenizers` wheel instead of falling back to a source build), and the usual plotting libraries.

In [ ]:
%pip install -q "themap[otdd] @ git+https://github.com/HFooladi/THEMAP.git" molfeat transformers matplotlib seaborn

Verify a GPU is actually attached before going further. If this prints `False`, change the runtime type and restart.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — switch runtime to T4 GPU and retry."
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device         : {torch.cuda.get_device_name(0)}")

## 2. Download a small sample dataset

We pull 3 source + 2 target ChEMBL assays directly from the THEMAP GitHub repo into the `datasets/train/` and `datasets/test/` layout the library expects. Each `*.jsonl.gz` is one assay with `SMILES` + binary `Property` labels.

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/HFooladi/THEMAP/main/datasets"
SAMPLES = {
    "train": ["CHEMBL3371729", "CHEMBL894522", "CHEMBL4224224"],
    "test": ["CHEMBL2219236", "CHEMBL2219358"],
}

for fold, ids in SAMPLES.items():
    os.makedirs(f"datasets/{fold}", exist_ok=True)
    for cid in ids:
        dst = f"datasets/{fold}/{cid}.jsonl.gz"
        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"{BASE}/{fold}/{cid}.jsonl.gz", dst)
        print(f"  {dst}  ({os.path.getsize(dst) / 1024:.1f} KB)")

## 3. Peek inside one dataset

Each `.jsonl.gz` becomes a `MoleculeDataset` — a list of SMILES plus a numpy array of binary labels.

In [ ]:
from themap import MoleculeDataset

ds = MoleculeDataset.load_from_file("datasets/test/CHEMBL2219236.jsonl.gz")
print(f"task_id        : {ds.task_id}")
print(f"#molecules     : {len(ds)}")
print(f"positive ratio : {ds.positive_ratio:.2%}")
print("first 3 SMILES :")
for s, y in zip(ds.smiles_list[:3], ds.labels[:3]):
    print(f"  [{y}] {s}")

## 4. How OTDD works in one paragraph

OTDD treats each dataset as a probability measure over `(feature_vector, label)` pairs. The distance between two datasets is the optimal-transport cost of moving the first measure onto the second under a composite ground cost:

- a **feature cost** — typically Euclidean in feature space — measures how far apart two molecules are;
- a **label cost** — the Wasserstein distance between the per-class feature distributions of the two datasets — measures how differently two labels behave.

THEMAP wraps this so you pass numpy arrays of features and labels; under the hood it calls into `themap.models.otdd` (a hardened fork of the reference implementation). For tractability THEMAP uses OTDD's Gaussian approximation for the inner label distances — i.e. it assumes the per-class feature distribution can be reasonably summarised by a mean and a covariance. **This assumption is the reason OTDD wants continuous features.** We will see why in section 6.

## 5. Compute OTDD once, end-to-end

The library exposes a single high-level entry point, `compute_dataset_distance_matrix`. We use `DatasetLoader` to discover the files we just downloaded and `MoleculeFeaturizer` to embed them. The same function handles `otdd`, `euclidean`, and `cosine` — you just pass `method=...`.

In [ ]:
import pandas as pd

from themap.data.loader import DatasetLoader
from themap.distance import compute_dataset_distance_matrix
from themap.features.molecule import MoleculeFeaturizer

loader = DatasetLoader("datasets")
src_ds = loader.load_datasets("train")  # {task_id: MoleculeDataset}
tgt_ds = loader.load_datasets("test")


def featurize_fold(datasets, featurizer):
    """Return (features_list, labels_list, ids_list) preserving dict order."""
    features_by_id = featurizer.featurize_datasets(datasets)
    ids = list(datasets.keys())
    return (
        [features_by_id[tid] for tid in ids],
        [datasets[tid].labels for tid in ids],
        ids,
    )


def compute_matrix(method, featurizer_name, maxsamples=500):
    featurizer = MoleculeFeaturizer(featurizer_name, device="cuda")
    src_feat, src_lab, src_ids = featurize_fold(src_ds, featurizer)
    tgt_feat, tgt_lab, tgt_ids = featurize_fold(tgt_ds, featurizer)
    result = compute_dataset_distance_matrix(
        source_features=src_feat,
        source_labels=src_lab,
        target_features=tgt_feat,
        target_labels=tgt_lab,
        source_ids=src_ids,
        target_ids=tgt_ids,
        method=method,
        device="cuda",
        maxsamples=maxsamples,
    )
    return pd.DataFrame(result).T.rename_axis(index="target", columns="source")


otdd_desc2D = compute_matrix(method="otdd", featurizer_name="desc2D")
print("OTDD with desc2D (RDKit 2D descriptors):")
otdd_desc2D.round(3)

## 6. Continuous vs discrete representations

Now we run the *same* OTDD computation with three very different featurizers:

| Featurizer | Type | Output |
|---|---|---|
| `ecfp` | Morgan fingerprint | **discrete**, 2048-bit binary vector |
| `desc2D` | RDKit 2D descriptors | **continuous**, ~200 real-valued descriptors |
| `ChemBERTa-77M-MTR` | Pretrained transformer | **continuous**, 384-d learned embedding |

OTDD's Gaussian inner-OT approximation summarises each class's feature distribution by `(mean, covariance)`. For continuous descriptors and learned embeddings this is reasonable: the features live in a smooth subspace of ℝ^d. For a 2048-bit binary fingerprint it is not: each bit is 0/1, covariance has limited meaning, classes share most bits, and the distance loses its ability to discriminate. The hypothesis: **OTDD with ECFP will produce a much narrower spread of distances than OTDD with desc2D or ChemBERTa.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FEATURIZERS = ["ecfp", "desc2D", "ChemBERTa-77M-MTR"]
otdd_by_feat = {f: compute_matrix("otdd", f) for f in FEATURIZERS}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, FEATURIZERS):
    sns.heatmap(
        otdd_by_feat[feat],
        annot=True,
        fmt=".2f",
        cmap="viridis_r",
        ax=ax,
        cbar=True,
    )
    ax.set_title(f"OTDD — {feat}")
plt.tight_layout()
plt.show()

print("\nDistance spread (max − min) per featurizer — wider is more discriminative:")
spread = {f: float(otdd_by_feat[f].values.max() - otdd_by_feat[f].values.min()) for f in FEATURIZERS}
for f, s in spread.items():
    print(f"  {f:22s}  spread = {s:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(spread.keys()), list(spread.values()), color=["#d95f02", "#1b9e77", "#7570b3"])
ax.set_ylabel("OTDD spread (max − min)")
ax.set_title("Discriminative power of OTDD across featurizers")
for i, (f, s) in enumerate(spread.items()):
    ax.text(i, s, f"{s:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

**What to look for.** The `desc2D` and `ChemBERTa-77M-MTR` panels should show clearly different distances between the two targets and their three candidate sources — a nearest-source for transfer learning emerges. The `ecfp` panel typically collapses: distances cluster in a narrow band, and the *ranking* of sources may not even agree with the continuous featurizers. That is the failure mode — OTDD still runs without error on fingerprints, but its output is not the signal you came for.

Practical rule of thumb when reaching for OTDD: pick `desc2D` (cheap, CPU-friendly to featurize), or a neural embedding (`ChemBERTa-77M-MTR`, `MolT5`, `gin_supervised_*`) if you want a smoother representation. Don't pass raw `ecfp`/`maccs`/`fcfp` to OTDD.

## 7. OTDD vs Euclidean vs Cosine on the same featurizer

Even with a continuous featurizer, OTDD is ~10–100× slower than Euclidean/Cosine on prototypes. What does it actually buy you? Both Euclidean and Cosine compute a distance between *class prototypes* (mean feature vector for positives, mean for negatives, concatenated) — they ignore the *shape* of each class's feature distribution and only look at where the means sit. OTDD looks at the full distribution.

Below: a 3 × 3 grid of distance matrices, with featurizers across rows and methods across columns. The matrices for prototype-based methods (Euclidean, Cosine) on fingerprints will look reasonable — those methods don't share OTDD's continuity assumption.

In [ ]:
METHODS = ["euclidean", "cosine", "otdd"]

grid = {(f, m): compute_matrix(m, f) for f in FEATURIZERS for m in METHODS}

fig, axes = plt.subplots(len(FEATURIZERS), len(METHODS), figsize=(13, 11))
for i, feat in enumerate(FEATURIZERS):
    for j, method in enumerate(METHODS):
        ax = axes[i, j]
        sns.heatmap(
            grid[(feat, method)],
            annot=True,
            fmt=".2f",
            cmap="viridis_r",
            ax=ax,
            cbar=False,
        )
        if i == 0:
            ax.set_title(method, fontsize=12, fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{feat}\n\ntarget", fontsize=10)
        else:
            ax.set_ylabel("")
        ax.set_xlabel("source" if i == len(FEATURIZERS) - 1 else "")
fig.suptitle("Distance matrices: featurizer (rows) × method (columns)", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, len(FEATURIZERS), figsize=(15, 4))
for ax, feat in zip(axes, FEATURIZERS):
    eucl = grid[(feat, "euclidean")].values.flatten()
    otdd = grid[(feat, "otdd")].values.flatten()
    rho, p = spearmanr(eucl, otdd)
    ax.scatter(eucl, otdd, s=80, color="#7570b3", edgecolor="black")
    for tgt in grid[(feat, "euclidean")].index:
        for src in grid[(feat, "euclidean")].columns:
            x = grid[(feat, "euclidean")].loc[tgt, src]
            y = grid[(feat, "otdd")].loc[tgt, src]
            ax.annotate(f"{tgt[-4:]}-{src[-4:]}", (x, y), fontsize=7, alpha=0.7)
    ax.set_xlabel("Euclidean (prototype)")
    ax.set_ylabel("OTDD")
    ax.set_title(f"{feat}\nSpearman ρ = {rho:.2f} (p={p:.2f})")
plt.tight_layout()
plt.show()

**Reading the scatter plots.** A high Spearman ρ means Euclidean and OTDD rank the source–target pairs the same way — OTDD's label-aware signal isn't telling you anything new on these data, and Euclidean is fine. A low or negative ρ means the two methods disagree, and OTDD's label-awareness is changing which source it would recommend for transfer learning. With three sources and two targets (six pairs), the correlation is noisy; the point of the exercise is to show that **the two methods are not interchangeable on continuous featurizers**, even when both run successfully.

## 8. Takeaways and where to go next

- **Use OTDD when transfer-learning source selection matters.** It captures label-aware signal that prototype-based Euclidean/Cosine miss.
- **Pair OTDD with continuous featurizers.** `desc2D` is a strong default; neural embeddings (`ChemBERTa-77M-MTR`, `MolT5`, `gin_supervised_*`) work too. Avoid raw binary fingerprints — OTDD still runs, but its output collapses.
- **For quick exploratory work, stick with Euclidean/Cosine + fingerprints** — the speed difference is large (seconds vs minutes on this notebook) and Euclidean is well-defined on discrete features.

### Same workflow from the command line

```bash
themap quick datasets/ -f desc2D -m otdd --device cuda -o output/
themap quick datasets/ -f ChemBERTa-77M-MTR -m otdd --device cuda -o output/
```

### Further reading

- [`01_quick_tour.ipynb`](01_quick_tour.ipynb) — the 5-minute intro with ECFP + Euclidean.
- [`02_api_deep_dive.ipynb`](02_api_deep_dive.ipynb) — the lower-level `DatasetDistance` and `MoleculeFeaturizer` APIs.
- THEMAP docs: [hfooladi.github.io/THEMAP](https://hfooladi.github.io/THEMAP/).
- Original OTDD paper: Alvarez-Melis & Fusi, *Geometric Dataset Distances via Optimal Transport*, NeurIPS 2020.
- THEMAP paper: Fooladi et al., *J. Chem. Inf. Model.* 64(10), 4031–4046 (2024).